In [ ]:
!pip install -q kaggle

In [ ]:
from google.colab import files
files.upload()   # Choose kaggle.json

Saving kaggle.json to kaggle.json


{'kaggle.json': b'{"username":"pollob","key":"d40ed3586538aed75b45a86651b42103"}'}

In [ ]:
!mkdir -p ~/.kaggle

In [ ]:
!mv kaggle.json ~/.kaggle/

In [ ]:
!chmod 600 ~/.kaggle/kaggle.json

In [ ]:
!kaggle datasets download -d pollob/custom-violence-prediction

Dataset URL: https://www.kaggle.com/datasets/pollob/custom-violence-prediction
License(s): unknown
100% 814M/814M [01:04<00:00, 13.3MB/s]



In [ ]:
!unzip -q *.zip -d ./data
!ls ./data

extracted_frames


In [ ]:
!pip install -q fvcore pytorchvideo av iopath

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.2/50.2 kB 5.4 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.7/132.7 kB 16.5 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.2/42.2 kB 5.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 36.4/36.4 MB 72.9 MB/s eta 0:00:00


In [ ]:
!pip install --upgrade timm

# PreVioNet (Proposed Model)

In [ ]:
import os
import numpy as np
import cv2
from pathlib import Path
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, roc_auc_score
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
import torchvision.models as tv_models

# ============================================
# ACTUAL XCEPTION + RESNET101 + BiLSTM
# ============================================

class XceptionResNet101BiLSTM(nn.Module):
    def __init__(self, num_classes=2, hidden_size=256, num_layers=2, dropout=0.5):
        super(XceptionResNet101BiLSTM, self).__init__()

        print("\nLoading Xception...")
        try:
            import timm
            self.xception = timm.create_model('xception', pretrained=True, num_classes=0)
            self.xception_dim = 2048
            self.use_timm = True
        except ImportError:
            self.xception = tv_models.resnet50(pretrained=True)
            self.xception = nn.Sequential(*list(self.xception.children())[:-2])
            self.xception_dim = 2048
            self.use_timm = False

        print("Loading ResNet101...")
        self.resnet101 = tv_models.resnet101(pretrained=True)
        self.resnet101 = nn.Sequential(*list(self.resnet101.children())[:-2])
        self.resnet_dim = 2048

        self.global_avg_pool = nn.AdaptiveAvgPool2d((1, 1))

        self.fusion = nn.Sequential(
            nn.Linear(4096, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(inplace=True),
            nn.Dropout(0.3)
        )

        self.bilstm = nn.LSTM(
            input_size=512,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            bidirectional=True,
            dropout=0.3 if num_layers > 1 else 0
        )

        self.temporal_attention = nn.Sequential(
            nn.Linear(hidden_size * 2, 128),
            nn.Tanh(),
            nn.Linear(128, 1)
        )

        self.classifier = nn.Sequential(
            nn.Linear(hidden_size * 2, 128),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout),
            nn.Linear(128, 64),
            nn.ReLU(inplace=True),
            nn.Dropout(0.3),
            nn.Linear(64, num_classes)
        )

    def forward(self, x):
        batch_size, seq_len, c, h, w = x.shape
        x = x.view(batch_size * seq_len, c, h, w)

        if self.use_timm:
            xception_features = self.xception(x)
        else:
            xception_features = self.xception(x)
            xception_features = self.global_avg_pool(xception_features)
        xception_features = xception_features.view(batch_size * seq_len, -1)

        resnet_features = self.resnet101(x)
        resnet_features = self.global_avg_pool(resnet_features)
        resnet_features = resnet_features.view(batch_size * seq_len, -1)

        combined = torch.cat([xception_features, resnet_features], dim=1)
        fused = self.fusion(combined)
        fused = fused.view(batch_size, seq_len, -1)

        lstm_output, _ = self.bilstm(fused)

        attn_weights = self.temporal_attention(lstm_output)
        attn_weights = F.softmax(attn_weights, dim=1)
        context = torch.sum(attn_weights * lstm_output, dim=1)

        output = self.classifier(context)
        return output


# ============================================
# DATASET
# ============================================

class ViolenceDataset(Dataset):
    def __init__(self, video_paths, labels, sequence_length=16, img_size=(299, 299)):
        self.video_paths = video_paths
        self.labels = labels
        self.sequence_length = sequence_length
        self.img_size = img_size

        self.transform = transforms.Compose([
            transforms.ToPILImage(),
            transforms.Resize(img_size),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
        ])

    def __len__(self):
        return len(self.video_paths)

    def load_frames(self, video_folder):
        frame_files = sorted(list(Path(video_folder).glob("*.jpg")) + list(Path(video_folder).glob("*.png")))
        frames = []
        for f in frame_files:
            img = cv2.imread(str(f))
            if img is not None:
                img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
                frames.append(img)
        return frames

    def sample_frames(self, frames):
        if len(frames) == 0:
            return [np.zeros((self.img_size[0], self.img_size[1], 3))] * self.sequence_length

        indices = np.linspace(0, len(frames) - 1, self.sequence_length, dtype=int)
        return [frames[i] for i in indices]

    def __getitem__(self, idx):
        frames = self.load_frames(self.video_paths[idx])
        sampled = self.sample_frames(frames)

        tensor_frames = [self.transform(f) for f in sampled]
        x = torch.stack(tensor_frames)
        y = torch.tensor(self.labels[idx], dtype=torch.long)

        return x, y


# ============================================
# TRAIN FOLD FUNCTION
# ============================================

def train_fold(model, train_loader, val_loader, epochs=30, lr=0.0001, device='cuda'):
    model = model.to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', patience=5, factor=0.5)

    best_val_acc = 0
    best_state = None

    for epoch in range(epochs):
        model.train()
        train_loss = 0
        train_correct = 0
        train_total = 0

        for data, labels in train_loader:
            data, labels = data.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(data)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            train_loss += loss.item()
            _, predicted = torch.max(outputs.data, 1)
            train_total += labels.size(0)
            train_correct += (predicted == labels).sum().item()

        train_acc = 100 * train_correct / train_total

        model.eval()
        val_loss = 0
        val_correct = 0
        val_total = 0

        with torch.no_grad():
            for data, labels in val_loader:
                data, labels = data.to(device), labels.to(device)
                outputs = model(data)
                loss = criterion(outputs, labels)

                val_loss += loss.item()
                _, predicted = torch.max(outputs.data, 1)
                val_total += labels.size(0)
                val_correct += (predicted == labels).sum().item()

        val_acc = 100 * val_correct / val_total
        scheduler.step(val_loss)

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}

    return best_state, best_val_acc


# ============================================
# EVALUATE FOLD
# ============================================

def evaluate_fold(model, test_loader, device='cuda'):
    model.eval()
    all_preds = []
    all_targets = []
    all_probs = []

    with torch.no_grad():
        for data, labels in test_loader:
            data, labels = data.to(device), labels.to(device)
            outputs = model(data)
            probs = F.softmax(outputs, dim=1)
            _, predicted = torch.max(outputs.data, 1)

            all_preds.extend(predicted.cpu().numpy())
            all_targets.extend(labels.cpu().numpy())
            all_probs.extend(probs.cpu().numpy()[:, 1])

    accuracy = accuracy_score(all_targets, all_preds)
    precision = precision_score(all_targets, all_preds, average=None, zero_division=0)
    recall = recall_score(all_targets, all_preds, average=None, zero_division=0)
    f1 = f1_score(all_targets, all_preds, average=None, zero_division=0)
    cm = confusion_matrix(all_targets, all_preds)
    auc = roc_auc_score(all_targets, all_probs)

    return accuracy, precision, recall, f1, cm, auc


# ============================================
# MAIN WITH 5-FOLD CV
# ============================================

if __name__ == "__main__":

    print("="*70)
    print("Xception + ResNet101 + BiLSTM - 5-FOLD CROSS VALIDATION")
    print("="*70)

    SEQUENCE_LENGTH = 16
    IMG_SIZE = (299, 299)
    BATCH_SIZE = 4
    EPOCHS = 30
    LEARNING_RATE = 0.0001

    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"\nUsing device: {device}")

    FRAMES_DIR = "/content/data/extracted_frames"

    violent_dir = os.path.join(FRAMES_DIR, "pre_fight_frames")
    non_violent_dir = os.path.join(FRAMES_DIR, "normal_frames")

    if os.path.exists(violent_dir) and os.path.exists(non_violent_dir):
        violent_videos = [str(p) for p in Path(violent_dir).glob("*") if p.is_dir()]
        non_violent_videos = [str(p) for p in Path(non_violent_dir).glob("*") if p.is_dir()]

        X = np.array(violent_videos + non_violent_videos)
        y = np.array([1] * len(violent_videos) + [0] * len(non_violent_videos))

        print(f"\nTotal violent videos: {len(violent_videos)}")
        print(f"Total non-violent videos: {len(non_violent_videos)}")
        print(f"Total videos: {len(X)}")

        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

        fold_accuracies = []
        fold_aucs = []
        all_probs = []
        all_labels = []

        for fold, (train_idx, test_idx) in enumerate(skf.split(X, y)):
            print(f"\n{'='*60}")
            print(f"FOLD {fold+1}/5")
            print(f"{'='*60}")

            X_train_fold = X[train_idx].tolist()
            X_test_fold = X[test_idx].tolist()
            y_train_fold = y[train_idx].tolist()
            y_test_fold = y[test_idx].tolist()

            # Split train into train/val (80-20)
            X_train, X_val, y_train, y_val = train_test_split(
                X_train_fold, y_train_fold, test_size=0.2, stratify=y_train_fold, random_state=42
            )

            print(f"Train: {len(X_train)} | Val: {len(X_val)} | Test: {len(X_test_fold)}")

            train_dataset = ViolenceDataset(X_train, y_train, SEQUENCE_LENGTH, IMG_SIZE)
            val_dataset = ViolenceDataset(X_val, y_val, SEQUENCE_LENGTH, IMG_SIZE)
            test_dataset = ViolenceDataset(X_test_fold, y_test_fold, SEQUENCE_LENGTH, IMG_SIZE)

            train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
            val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
            test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

            model = XceptionResNet101BiLSTM(num_classes=2)

            best_state, best_val_acc = train_fold(model, train_loader, val_loader, EPOCHS, LEARNING_RATE, device)

            model.load_state_dict(best_state)
            acc, prec, rec, f1, cm, auc = evaluate_fold(model, test_loader, device)

            fold_accuracies.append(acc)
            fold_aucs.append(auc)

            print(f"\nFold {fold+1} Results:")
            print(f"  Accuracy: {acc*100:.2f}%")
            print(f"  AUC: {auc:.4f}")
            print(f"  Confusion Matrix: TN={cm[0,0]}, FP={cm[0,1]}, FN={cm[1,0]}, TP={cm[1,1]}")

        # ============================================
        # CROSS VALIDATION SUMMARY
        # ============================================

        print("\n" + "=" * 70)
        print("5-FOLD CROSS VALIDATION SUMMARY")
        print("=" * 70)

        mean_acc = np.mean(fold_accuracies)
        std_acc = np.std(fold_accuracies)
        mean_auc = np.mean(fold_aucs)
        std_auc = np.std(fold_aucs)

        print(f"\nAccuracy: {mean_acc:.4f} ± {std_acc:.4f} ({mean_acc*100:.2f}% ± {std_acc*100:.2f}%)")
        print(f"AUC: {mean_auc:.4f} ± {std_auc:.4f}")

        print(f"\nPer-fold Accuracies: {[f'{acc*100:.2f}%' for acc in fold_accuracies]}")
        print(f"Per-fold AUCs: {[f'{auc:.4f}' for auc in fold_aucs]}")

        print("\n" + "=" * 70)
        print("CROSS VALIDATION COMPLETE")
        print("=" * 70)

    else:
        print(f"\nDataset not found at: {FRAMES_DIR}")

Xception + ResNet101 + BiLSTM - 5-FOLD CROSS VALIDATION

Using device: cuda

Total violent videos: 95
Total non-violent videos: 84
Total videos: 179

FOLD 1/5
Train: 114 | Val: 29 | Test: 36

Loading Xception...


/usr/local/lib/python3.12/dist-packages/timm/models/_factory.py:138: UserWarning: Mapping deprecated model name xception to current legacy_xception.
  model = create_fn(


Downloading: "https://github.com/rwightman/pytorch-image-models/releases/download/v0.1-cadene/xception-43020ad28.pth" to /root/.cache/torch/hub/checkpoints/xception-43020ad28.pth
Loading ResNet101...


/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet101_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet101_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Downloading: "https://download.pytorch.org/models/resnet101-63fe2227.pth" to /root/.cache/torch/hub/checkpoints/resnet101-63fe2227.pth


...


Fold 4 Results:
  Accuracy: 88.89%
  AUC: 0.9690
  Confusion Matrix: TN=14, FP=3, FN=1, TP=18

FOLD 5/5
Train: 115 | Val: 29 | Test: 35

Loading Xception...


/usr/local/lib/python3.12/dist-packages/timm/models/_factory.py:138: UserWarning: Mapping deprecated model name xception to current legacy_xception.
  model = create_fn(


Loading ResNet101...


/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet101_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet101_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)



Fold 5 Results:
  Accuracy: 97.14%
  AUC: 1.0000
  Confusion Matrix: TN=15, FP=1, FN=0, TP=19

5-FOLD CROSS VALIDATION SUMMARY

Accuracy: 0.9332 ± 0.0281 (93.32% ± 2.81%)
AUC: 0.9684 ± 0.0245

Per-fold Accuracies: ['94.44%', '94.44%', '91.67%', '88.89%', '97.14%']
Per-fold AUCs: ['0.9443', '0.9381', '0.9907', '0.9690', '1.0000']

CROSS VALIDATION COMPLETE


# HYBRID MODEL: MobileNetV3-GRU + EfficientNetB3-GRU + FUSION + ATTENTION 5-FOLD STRATIFIED CROSS VALIDATION

In [ ]:
import os
import numpy as np
import cv2
from pathlib import Path

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, roc_auc_score

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models


# ============================================
# COMPLETE MODEL: MobileNetV3-GRU + EfficientNetB3-GRU + FUSION + ATTENTION
# ============================================

class HybridViolenceDetector(nn.Module):
    def __init__(self, num_classes=2, hidden_size=256):
        super().__init__()

        print("=" * 60)
        print("LOADING HYBRID MODEL")
        print("=" * 60)

        # ============================================
        # BRANCH 1: MobileNetV3 + GRU
        # ============================================
        print("\n1. Loading MobileNetV3...")
        self.mobilenet = models.mobilenet_v3_large(weights=models.MobileNet_V3_Large_Weights.DEFAULT)
        self.mobilenet.classifier = nn.Identity()
        self.mobilenet_dim = 960

        self.mobilenet_gru = nn.GRU(
            input_size=self.mobilenet_dim,
            hidden_size=hidden_size,
            batch_first=True,
            bidirectional=True,
            num_layers=2,
            dropout=0.3
        )

        # ============================================
        # BRANCH 2: EfficientNetB3 + GRU
        # ============================================
        print("\n2. Loading EfficientNetB3...")
        self.efficientnet = models.efficientnet_b3(weights=models.EfficientNet_B3_Weights.DEFAULT)
        self.efficientnet.classifier = nn.Identity()
        self.efficientnet_dim = 1536

        self.efficientnet_gru = nn.GRU(
            input_size=self.efficientnet_dim,
            hidden_size=hidden_size,
            batch_first=True,
            bidirectional=True,
            num_layers=2,
            dropout=0.3
        )

        # ============================================
        # FUSION LAYER
        # ============================================
        self.fusion_dim = (hidden_size * 2) + (hidden_size * 2)

        print(f"\nMobileNetV3 GRU output: {hidden_size * 2}")
        print(f"EfficientNetB3 GRU output: {hidden_size * 2}")
        print(f"Fusion dimension: {self.fusion_dim}")

        self.fusion = nn.Sequential(
            nn.Linear(self.fusion_dim, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(),
            nn.Dropout(0.3)
        )

        # ============================================
        # TEMPORAL ATTENTION
        # ============================================
        self.attention = nn.Sequential(
            nn.Linear(512, 128),
            nn.Tanh(),
            nn.Linear(128, 1)
        )

        # ============================================
        # CLASSIFIER
        # ============================================
        self.classifier = nn.Sequential(
            nn.Linear(512, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(64, num_classes)
        )

        print(f"\n✓ Model built successfully!")
        print(f"  Total parameters: {sum(p.numel() for p in self.parameters()):,}")
        print("=" * 60)

    def forward(self, x):
        B, T, C, H, W = x.shape
        x = x.view(B * T, C, H, W)

        mobilenet_features = self.mobilenet(x)
        if len(mobilenet_features.shape) == 4:
            mobilenet_features = F.adaptive_avg_pool2d(mobilenet_features, 1)
            mobilenet_features = mobilenet_features.view(B * T, -1)
        mobilenet_features = mobilenet_features.view(B, T, -1)
        mobilenet_out, _ = self.mobilenet_gru(mobilenet_features)
        mobilenet_out = mobilenet_out[:, -1, :]

        efficientnet_features = self.efficientnet(x)
        if len(efficientnet_features.shape) == 4:
            efficientnet_features = F.adaptive_avg_pool2d(efficientnet_features, 1)
            efficientnet_features = efficientnet_features.view(B * T, -1)
        efficientnet_features = efficientnet_features.view(B, T, -1)
        efficientnet_out, _ = self.efficientnet_gru(efficientnet_features)
        efficientnet_out = efficientnet_out[:, -1, :]

        combined = torch.cat([mobilenet_out, efficientnet_out], dim=1)
        fused = self.fusion(combined)

        fused_seq = fused.unsqueeze(1)
        attn_weights = self.attention(fused_seq)
        attn_weights = F.softmax(attn_weights, dim=1)
        context = torch.sum(attn_weights * fused_seq, dim=1)

        output = self.classifier(context)
        return output


# ============================================
# DATASET
# ============================================

class ViolenceDataset(Dataset):
    def __init__(self, video_paths, labels, seq_len=16, img_size=(224, 224)):
        self.video_paths = video_paths
        self.labels = labels
        self.seq_len = seq_len
        self.img_size = img_size

        self.transform = transforms.Compose([
            transforms.ToPILImage(),
            transforms.Resize(img_size),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
        ])

    def load_frames(self, folder):
        frames = []
        for f in sorted(Path(folder).glob("*.jpg")):
            img = cv2.imread(str(f))
            if img is not None:
                img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
                frames.append(img)
        return frames

    def sample(self, frames):
        if len(frames) == 0:
            return [np.zeros((self.img_size[0], self.img_size[1], 3), dtype=np.uint8)] * self.seq_len
        idx = np.linspace(0, len(frames) - 1, self.seq_len).astype(int)
        return [frames[i] for i in idx]

    def __len__(self):
        return len(self.video_paths)

    def __getitem__(self, idx):
        frames = self.load_frames(self.video_paths[idx])
        frames = self.sample(frames)
        frames = [self.transform(f) for f in frames]
        x = torch.stack(frames)
        y = torch.tensor(self.labels[idx], dtype=torch.long)
        return x, y


# ============================================
# TRAIN FUNCTION FOR A SINGLE FOLD
# ============================================

def train_fold(model, train_loader, val_loader, device, epochs=20):
    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-5)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', patience=3, factor=0.5)
    criterion = nn.CrossEntropyLoss()

    best_val_acc = 0
    best_state_dict = None

    for epoch in range(epochs):
        model.train()
        train_loss, train_correct, total = 0, 0, 0

        for x, y in train_loader:
            x, y = x.to(device), y.to(device)
            optimizer.zero_grad()
            out = model(x)
            loss = criterion(out, y)
            loss.backward()
            optimizer.step()

            train_loss += loss.item()
            train_correct += (out.argmax(1) == y).sum().item()
            total += y.size(0)

        train_acc = train_correct / total
        train_loss = train_loss / len(train_loader)

        model.eval()
        val_loss, val_correct, val_total = 0, 0, 0

        with torch.no_grad():
            for x, y in val_loader:
                x, y = x.to(device), y.to(device)
                out = model(x)
                loss = criterion(out, y)

                val_loss += loss.item()
                val_correct += (out.argmax(1) == y).sum().item()
                val_total += y.size(0)

        val_acc = val_correct / val_total
        val_loss = val_loss / len(val_loader)

        scheduler.step(val_loss)

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_state_dict = {k: v.cpu().clone() for k, v in model.state_dict().items()}

    return best_state_dict, best_val_acc


# ============================================
# EVALUATE FOLD
# ============================================

def evaluate_fold(model, test_loader, device):
    model.eval()
    preds, labels, probs = [], [], []

    with torch.no_grad():
        for x, y in test_loader:
            x, y = x.to(device), y.to(device)
            out = model(x)
            p = F.softmax(out, dim=1)

            preds.extend(out.argmax(1).cpu().numpy())
            labels.extend(y.cpu().numpy())
            probs.extend(p[:, 1].cpu().numpy())

    acc = accuracy_score(labels, preds)
    prec = precision_score(labels, preds, average=None, zero_division=0)
    rec = recall_score(labels, preds, average=None, zero_division=0)
    f1 = f1_score(labels, preds, average=None, zero_division=0)
    cm = confusion_matrix(labels, preds)
    auc = roc_auc_score(labels, probs)

    return acc, prec, rec, f1, cm, auc, probs, labels


# ============================================
# MAIN WITH 5-FOLD CROSS VALIDATION
# ============================================

if __name__ == "__main__":

    DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    print("=" * 60)
    print("HYBRID MODEL: MobileNetV3-GRU + EfficientNetB3-GRU + FUSION + ATTENTION")
    print("5-FOLD STRATIFIED CROSS VALIDATION")
    print("=" * 60)
    print(f"Using device: {DEVICE}")

    FRAMES_DIR = "/content/data/extracted_frames"

    violent_dir = os.path.join(FRAMES_DIR, "pre_fight_frames")
    normal_dir = os.path.join(FRAMES_DIR, "normal_frames")

    violent = [str(p) for p in Path(violent_dir).glob("*") if p.is_dir()]
    normal = [str(p) for p in Path(normal_dir).glob("*") if p.is_dir()]

    X = violent + normal
    y = [1] * len(violent) + [0] * len(normal)

    print(f"\nTotal violent videos: {len(violent)}")
    print(f"Total non-violent videos: {len(normal)}")
    print(f"Total videos: {len(X)}")

    X = np.array(X)
    y = np.array(y)

    # 5-Fold Stratified Cross Validation
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

    fold_accuracies = []
    fold_aucs = []
    fold_precisions = []
    fold_recalls = []
    fold_f1s = []
    all_probs = []
    all_labels = []

    for fold, (train_idx, test_idx) in enumerate(skf.split(X, y)):
        print(f"\n{'='*60}")
        print(f"FOLD {fold+1}/5")
        print(f"{'='*60}")

        X_train_fold = X[train_idx].tolist()
        X_test_fold = X[test_idx].tolist()
        y_train_fold = y[train_idx].tolist()
        y_test_fold = y[test_idx].tolist()

        # Split train into train/val (80-20)
        from sklearn.model_selection import train_test_split
        X_train, X_val, y_train, y_val = train_test_split(
            X_train_fold, y_train_fold, test_size=0.2, stratify=y_train_fold, random_state=42
        )

        print(f"Train: {len(X_train)} | Val: {len(X_val)} | Test: {len(X_test_fold)}")

        train_ds = ViolenceDataset(X_train, y_train)
        val_ds = ViolenceDataset(X_val, y_val)
        test_ds = ViolenceDataset(X_test_fold, y_test_fold)

        train_loader = DataLoader(train_ds, batch_size=4, shuffle=True, num_workers=2)
        val_loader = DataLoader(val_ds, batch_size=4, shuffle=False, num_workers=2)
        test_loader = DataLoader(test_ds, batch_size=4, shuffle=False, num_workers=2)

        model = HybridViolenceDetector(num_classes=2, hidden_size=256)
        model = model.to(DEVICE)

        # Train fold
        best_state_dict, best_val_acc = train_fold(model, train_loader, val_loader, DEVICE, epochs=20)
        model.load_state_dict(best_state_dict)

        # Evaluate fold
        acc, prec, rec, f1, cm, auc, probs, labels = evaluate_fold(model, test_loader, DEVICE)

        fold_accuracies.append(acc)
        fold_aucs.append(auc)
        fold_precisions.append(prec)
        fold_recalls.append(rec)
        fold_f1s.append(f1)
        all_probs.extend(probs)
        all_labels.extend(labels)

        print(f"\nFold {fold+1} Results:")
        print(f"  Accuracy: {acc:.4f} ({acc*100:.2f}%)")
        print(f"  AUC: {auc:.4f}")
        print(f"  Precision (Normal, Violence): {prec[0]:.4f}, {prec[1]:.4f}")
        print(f"  Recall (Normal, Violence): {rec[0]:.4f}, {rec[1]:.4f}")
        print(f"  Confusion Matrix: TN={cm[0,0]}, FP={cm[0,1]}, FN={cm[1,0]}, TP={cm[1,1]}")

    # ============================================
    # CROSS VALIDATION SUMMARY
    # ============================================

    print("\n" + "=" * 60)
    print("5-FOLD CROSS VALIDATION SUMMARY")
    print("=" * 60)

    mean_acc = np.mean(fold_accuracies)
    std_acc = np.std(fold_accuracies)
    mean_auc = np.mean(fold_aucs)
    std_auc = np.std(fold_aucs)

    print(f"\nAccuracy: {mean_acc:.4f} ± {std_acc:.4f} ({mean_acc*100:.2f}% ± {std_acc*100:.2f}%)")
    print(f"AUC: {mean_auc:.4f} ± {std_auc:.4f}")

    print(f"\nPer-fold Accuracies: {[f'{acc:.4f}' for acc in fold_accuracies]}")
    print(f"Per-fold AUCs: {[f'{auc:.4f}' for auc in fold_aucs]}")

    # Overall metrics from all predictions
    overall_acc = accuracy_score(all_labels, [1 if p > 0.5 else 0 for p in all_probs])
    overall_auc = roc_auc_score(all_labels, all_probs)

    print(f"\nOverall Accuracy (all predictions): {overall_acc:.4f} ({overall_acc*100:.2f}%)")
    print(f"Overall AUC (all predictions): {overall_auc:.4f}")

    print("\n" + "=" * 60)
    print("CROSS VALIDATION COMPLETE")
    print("=" * 60)

HYBRID MODEL: MobileNetV3-GRU + EfficientNetB3-GRU + FUSION + ATTENTION
5-FOLD STRATIFIED CROSS VALIDATION
Using device: cuda

Total violent videos: 95
Total non-violent videos: 84
Total videos: 179

FOLD 1/5
Train: 114 | Val: 29 | Test: 36
LOADING HYBRID MODEL

1. Loading MobileNetV3...

2. Loading EfficientNetB3...

MobileNetV3 GRU output: 512
EfficientNetB3 GRU output: 512
Fusion dimension: 1024

✓ Model built successfully!
  Total parameters: 21,325,723

Fold 1 Results:
  Accuracy: 0.8889 (88.89%)
  AUC: 0.9628
  Precision (Normal, Violence): 0.8824, 0.8947
  Recall (Normal, Violence): 0.8824, 0.8947
  Confusion Matrix: TN=15, FP=2, FN=2, TP=17

FOLD 2/5
Train: 114 | Val: 29 | Test: 36
LOADING HYBRID MODEL

1. Loading MobileNetV3...

2. Loading EfficientNetB3...

MobileNetV3 GRU output: 512
EfficientNetB3 GRU output: 512
Fusion dimension: 1024

✓ Model built successfully!
  Total parameters: 21,325,723

Fold 2 Results:
  Accuracy: 0.8333 (83.33%)
  AUC: 0.8978
  Precision (Normal, 

## INCEPTION V3 + BiLSTM (Cross Validaition)

In [ ]:
import os
import numpy as np
import cv2
from pathlib import Path
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, roc_auc_score
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models

# ============================================
# INCEPTION V3 + BiLSTM (FULL FIXED VERSION)
# ============================================

class InceptionV3BiLSTM(nn.Module):
    def __init__(self, num_classes=2, hidden_size=256, num_layers=2, dropout=0.5):
        super(InceptionV3BiLSTM, self).__init__()

        print("Loading InceptionV3...")

        self.inception = models.inception_v3(pretrained=True, aux_logits=True, transform_input=False)
        self.inception.fc = nn.Identity()

        self.feature_dim = 2048

        self.bilstm = nn.LSTM(
            input_size=self.feature_dim,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            bidirectional=True,
            dropout=dropout if num_layers > 1 else 0
        )

        self.attention = nn.Sequential(
            nn.Linear(hidden_size * 2, 128),
            nn.Tanh(),
            nn.Linear(128, 1)
        )

        self.classifier = nn.Sequential(
            nn.Linear(hidden_size * 2, 128),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(64, num_classes)
        )

    def forward(self, x):
        b, t, c, h, w = x.shape
        x = x.view(b * t, c, h, w)
        features = self.inception(x)
        if isinstance(features, tuple):
            features = features[0]
        features = features.view(b, t, -1)
        lstm_out, _ = self.bilstm(features)
        attn = self.attention(lstm_out)
        attn = F.softmax(attn, dim=1)
        context = torch.sum(attn * lstm_out, dim=1)
        return self.classifier(context)


# ============================================
# DATASET
# ============================================

class ViolenceDataset(Dataset):
    def __init__(self, video_paths, labels, sequence_length=16, img_size=(299, 299)):
        self.video_paths = video_paths
        self.labels = labels
        self.sequence_length = sequence_length
        self.img_size = img_size

        self.transform = transforms.Compose([
            transforms.ToPILImage(),
            transforms.Resize(img_size),
            transforms.ToTensor(),
            transforms.Normalize([0.485, 0.456, 0.406],
                                 [0.229, 0.224, 0.225])
        ])

    def __len__(self):
        return len(self.video_paths)

    def load_frames(self, video_folder):
        frame_files = sorted(list(Path(video_folder).glob("*.jpg")) +
                             list(Path(video_folder).glob("*.png")))
        frames = []
        for f in frame_files:
            img = cv2.imread(str(f))
            if img is not None:
                img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
                frames.append(img)
        return frames

    def sample_frames(self, frames):
        if len(frames) == 0:
            return [np.zeros((self.img_size[0], self.img_size[1], 3))] * self.sequence_length
        idx = np.linspace(0, len(frames) - 1, self.sequence_length, dtype=int)
        return [frames[i] for i in idx]

    def __getitem__(self, idx):
        frames = self.load_frames(self.video_paths[idx])
        frames = self.sample_frames(frames)
        frames = [self.transform(f) for f in frames]
        x = torch.stack(frames)
        y = torch.tensor(self.labels[idx], dtype=torch.long)
        return x, y


# ============================================
# TRAIN FOLD FUNCTION
# ============================================

def train_fold(model, train_loader, val_loader, epochs=10, lr=1e-4, device='cuda'):
    model = model.to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr)
    best_acc = 0
    best_state = None

    for epoch in range(epochs):
        model.train()
        train_correct = 0
        train_total = 0

        for x, y in train_loader:
            x, y = x.to(device), y.to(device)
            optimizer.zero_grad()
            out = model(x)
            loss = criterion(out, y)
            loss.backward()
            optimizer.step()
            pred = torch.argmax(out, 1)
            train_correct += (pred == y).sum().item()
            train_total += y.size(0)

        train_acc = train_correct / train_total

        model.eval()
        val_correct = 0
        val_total = 0

        with torch.no_grad():
            for x, y in val_loader:
                x, y = x.to(device), y.to(device)
                out = model(x)
                pred = torch.argmax(out, 1)
                val_correct += (pred == y).sum().item()
                val_total += y.size(0)

        val_acc = val_correct / val_total

        if val_acc > best_acc:
            best_acc = val_acc
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}

    return best_state, best_acc


# ============================================
# EVALUATE FOLD
# ============================================

def evaluate_fold(model, test_loader, device='cuda'):
    model.eval()
    preds, targets, probs = [], [], []

    with torch.no_grad():
        for x, y in test_loader:
            x, y = x.to(device), y.to(device)
            out = model(x)
            p = F.softmax(out, dim=1)
            preds.extend(torch.argmax(out, 1).cpu().numpy())
            targets.extend(y.cpu().numpy())
            probs.extend(p[:, 1].cpu().numpy())

    acc = accuracy_score(targets, preds)
    prec = precision_score(targets, preds, average=None, zero_division=0)
    rec = recall_score(targets, preds, average=None, zero_division=0)
    f1 = f1_score(targets, preds, average=None, zero_division=0)
    cm = confusion_matrix(targets, preds)
    auc = roc_auc_score(targets, probs)

    return acc, prec, rec, f1, cm, auc, probs, targets


# ============================================
# MAIN WITH 5-FOLD CV
# ============================================

if __name__ == "__main__":

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Using device: {device}")

    FRAMES_DIR = "/content/data/extracted_frames"

    violent = [str(p) for p in Path(FRAMES_DIR + "/pre_fight_frames").glob("*")]
    normal = [str(p) for p in Path(FRAMES_DIR + "/normal_frames").glob("*")]

    X = np.array(violent + normal)
    y = np.array([1] * len(violent) + [0] * len(normal))

    print(f"Total violent videos: {len(violent)}")
    print(f"Total normal videos: {len(normal)}")
    print(f"Total videos: {len(X)}")

    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

    fold_accuracies = []
    fold_aucs = []
    all_probs = []
    all_labels = []

    for fold, (train_idx, test_idx) in enumerate(skf.split(X, y)):
        print(f"\n{'='*60}")
        print(f"FOLD {fold+1}/5")
        print(f"{'='*60}")

        X_train_fold = X[train_idx].tolist()
        X_test_fold = X[test_idx].tolist()
        y_train_fold = y[train_idx].tolist()
        y_test_fold = y[test_idx].tolist()

        # Split train into train/val (80-20)
        from sklearn.model_selection import train_test_split
        X_train, X_val, y_train, y_val = train_test_split(
            X_train_fold, y_train_fold, test_size=0.2, stratify=y_train_fold, random_state=42
        )

        print(f"Train: {len(X_train)} | Val: {len(X_val)} | Test: {len(X_test_fold)}")

        train_ds = ViolenceDataset(X_train, y_train)
        val_ds = ViolenceDataset(X_val, y_val)
        test_ds = ViolenceDataset(X_test_fold, y_test_fold)

        train_loader = DataLoader(train_ds, batch_size=4, shuffle=True)
        val_loader = DataLoader(val_ds, batch_size=4, shuffle=False)
        test_loader = DataLoader(test_ds, batch_size=4, shuffle=False)

        model = InceptionV3BiLSTM()
        best_state, best_val_acc = train_fold(model, train_loader, val_loader, epochs=10, device=device)

        model.load_state_dict(best_state)
        acc, prec, rec, f1, cm, auc, probs, labels = evaluate_fold(model, test_loader, device)

        fold_accuracies.append(acc)
        fold_aucs.append(auc)
        all_probs.extend(probs)
        all_labels.extend(labels)

        print(f"\nFold {fold+1} Results:")
        print(f"  Accuracy: {acc:.4f} ({acc*100:.2f}%)")
        print(f"  AUC: {auc:.4f}")
        print(f"  Confusion Matrix: TN={cm[0,0]}, FP={cm[0,1]}, FN={cm[1,0]}, TP={cm[1,1]}")

    # ============================================
    # CROSS VALIDATION SUMMARY
    # ============================================

    print("\n" + "=" * 60)
    print("5-FOLD CROSS VALIDATION SUMMARY")
    print("=" * 60)

    mean_acc = np.mean(fold_accuracies)
    std_acc = np.std(fold_accuracies)
    mean_auc = np.mean(fold_aucs)
    std_auc = np.std(fold_aucs)

    print(f"\nAccuracy: {mean_acc:.4f} ± {std_acc:.4f} ({mean_acc*100:.2f}% ± {std_acc*100:.2f}%)")
    print(f"AUC: {mean_auc:.4f} ± {std_auc:.4f}")

    print(f"\nPer-fold Accuracies: {[f'{acc:.4f}' for acc in fold_accuracies]}")
    print(f"Per-fold AUCs: {[f'{auc:.4f}' for auc in fold_aucs]}")

    overall_acc = accuracy_score(all_labels, [1 if p > 0.5 else 0 for p in all_probs])
    overall_auc = roc_auc_score(all_labels, all_probs)

    print(f"\nOverall Accuracy (all predictions): {overall_acc:.4f} ({overall_acc*100:.2f}%)")
    print(f"Overall AUC (all predictions): {overall_auc:.4f}")

    print("\n" + "=" * 60)
    print("CROSS VALIDATION COMPLETE")
    print("=" * 60)

Using device: cuda
Total violent videos: 95
Total normal videos: 84
Total videos: 179

FOLD 1/5
Train: 114 | Val: 29 | Test: 36
Loading InceptionV3...
Downloading: "https://download.pytorch.org/models/inception_v3_google-0cc3c7bd.pth" to /root/.cache/torch/hub/checkpoints/inception_v3_google-0cc3c7bd.pth


/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=Inception_V3_Weights.IMAGENET1K_V1`. You can also use `weights=Inception_V3_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
100%|██████████| 104M/104M [00:00<00:00, 175MB/s] 



Fold 1 Results:
  Accuracy: 0.8611 (86.11%)
  AUC: 0.8824
  Confusion Matrix: TN=14, FP=3, FN=2, TP=17

FOLD 2/5
Train: 114 | Val: 29 | Test: 36
Loading InceptionV3...


/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=Inception_V3_Weights.IMAGENET1K_V1`. You can also use `weights=Inception_V3_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)



Fold 2 Results:
  Accuracy: 0.7778 (77.78%)
  AUC: 0.8947
  Confusion Matrix: TN=13, FP=4, FN=4, TP=15

FOLD 3/5
Train: 114 | Val: 29 | Test: 36
Loading InceptionV3...


...


Fold 3 Results:
  Accuracy: 0.8889 (88.89%)
  AUC: 0.9009
  Confusion Matrix: TN=15, FP=2, FN=2, TP=17

FOLD 4/5
Train: 114 | Val: 29 | Test: 36
Loading InceptionV3...


/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=Inception_V3_Weights.IMAGENET1K_V1`. You can also use `weights=Inception_V3_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)



Fold 4 Results:
  Accuracy: 0.8611 (86.11%)
  AUC: 0.9350
  Confusion Matrix: TN=15, FP=2, FN=3, TP=16

FOLD 5/5
Train: 115 | Val: 29 | Test: 35
Loading InceptionV3...


/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=Inception_V3_Weights.IMAGENET1K_V1`. You can also use `weights=Inception_V3_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)



Fold 5 Results:
  Accuracy: 0.9143 (91.43%)
  AUC: 0.9507
  Confusion Matrix: TN=13, FP=3, FN=0, TP=19

5-FOLD CROSS VALIDATION SUMMARY

Accuracy: 0.8606 ± 0.0459 (86.06% ± 4.59%)
AUC: 0.9127 ± 0.0258

Per-fold Accuracies: ['0.8611', '0.7778', '0.8889', '0.8611', '0.9143']
Per-fold AUCs: ['0.8824', '0.8947', '0.9009', '0.9350', '0.9507']

Overall Accuracy (all predictions): 0.8603 (86.03%)
Overall AUC (all predictions): 0.9054

CROSS VALIDATION COMPLETE


# **MobilenetV3-positional encoding-ViT\** cross validation

In [ ]:
import os
import numpy as np
import cv2
from pathlib import Path
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
import math

# ============================================
# POSITIONAL ENCODER
# ============================================

class PositionalEncoder(nn.Module):
    def __init__(self, d_model, max_len=500):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len).unsqueeze(1).float()
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer('pe', pe.unsqueeze(0))

    def forward(self, x):
        return x + self.pe[:, :x.size(1)]

# ============================================
# MODEL
# ============================================

class MobileNetV3_ViT(nn.Module):
    def __init__(self, num_classes=2):
        super().__init__()

        import timm

        self.mobilenet = timm.create_model('mobilenetv3_large_100', pretrained=True, num_classes=0)
        self.vit = timm.create_model('vit_base_patch16_224', pretrained=True)

        self.feature_dim = 1280 + 768

        self.pos_encoder = PositionalEncoder(self.feature_dim)

        self.classifier = nn.Sequential(
            nn.Linear(self.feature_dim, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, num_classes)
        )

    def forward(self, x):
        b, t, c, h, w = x.shape
        x = x.view(b * t, c, h, w)

        mob = self.mobilenet(x)
        if mob.dim() == 4:
            mob = F.adaptive_avg_pool2d(mob, 1)
            mob = mob.view(mob.size(0), -1)

        vit = self.vit.forward_features(x)
        if isinstance(vit, tuple):
            vit = vit[0]
        if vit.dim() == 3:
            vit = vit[:, 0]

        feat = torch.cat([mob, vit], dim=1)
        feat = feat.view(b, t, -1)

        feat = self.pos_encoder(feat)
        feat = feat.mean(dim=1)

        return self.classifier(feat)

# ============================================
# DATASET
# ============================================

class ViolenceDataset(Dataset):
    def __init__(self, video_paths, labels, sequence_length=8, img_size=(224, 224)):
        self.video_paths = video_paths
        self.labels = labels
        self.sequence_length = sequence_length

        self.transform = transforms.Compose([
            transforms.ToPILImage(),
            transforms.Resize(img_size),
            transforms.ToTensor(),
            transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])
        ])

    def __len__(self):
        return len(self.video_paths)

    def load_frames(self, folder):
        frames = []
        for f in sorted(Path(folder).glob("*.jpg")):
            img = cv2.imread(str(f))
            if img is not None:
                img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
                frames.append(img)
        return frames

    def __getitem__(self, idx):
        frames = self.load_frames(self.video_paths[idx])

        if len(frames) == 0:
            frames = [np.zeros((224,224,3))]*self.sequence_length

        idxs = np.linspace(0, len(frames)-1, self.sequence_length).astype(int)
        frames = [frames[i] for i in idxs]

        frames = torch.stack([self.transform(f) for f in frames])
        label = torch.tensor(self.labels[idx], dtype=torch.long)

        return frames, label

# ============================================
# TRAIN FOLD FUNCTION
# ============================================

def train_fold(model, train_loader, val_loader, device, epochs=20):
    model = model.to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=1e-4)
    crit = nn.CrossEntropyLoss()

    best_val_acc = 0
    best_state = None

    for epoch in range(epochs):
        model.train()
        train_loss = 0
        train_correct = 0
        train_total = 0

        for data, labels in train_loader:
            data, labels = data.to(device), labels.to(device)

            opt.zero_grad()
            out = model(data)
            loss = crit(out, labels)
            loss.backward()
            opt.step()

            train_loss += loss.item()
            _, pred = torch.max(out, 1)
            train_total += labels.size(0)
            train_correct += (pred == labels).sum().item()

        train_acc = train_correct / train_total

        model.eval()
        val_loss = 0
        val_correct = 0
        val_total = 0

        with torch.no_grad():
            for data, labels in val_loader:
                data, labels = data.to(device), labels.to(device)
                out = model(data)
                loss = crit(out, labels)

                val_loss += loss.item()
                _, pred = torch.max(out, 1)
                val_total += labels.size(0)
                val_correct += (pred == labels).sum().item()

        val_acc = val_correct / val_total

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}

    return best_state, best_val_acc

# ============================================
# EVALUATE FOLD
# ============================================

def evaluate_fold(model, loader, device):
    model.eval()
    preds, targets = [], []

    with torch.no_grad():
        for data, labels in loader:
            data, labels = data.to(device), labels.to(device)
            out = model(data)
            _, pred = torch.max(out, 1)

            preds.extend(pred.cpu().numpy())
            targets.extend(labels.cpu().numpy())

    acc = accuracy_score(targets, preds)
    prec = precision_score(targets, preds, zero_division=0)
    rec = recall_score(targets, preds, zero_division=0)
    f1 = f1_score(targets, preds, zero_division=0)
    cm = confusion_matrix(targets, preds)

    return acc, prec, rec, f1, cm

# ============================================
# MAIN WITH 5-FOLD CV
# ============================================

if __name__ == "__main__":

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Using device: {device}")

    FRAMES_DIR = "/content/data/extracted_frames"

    violent = list(Path(FRAMES_DIR+"/pre_fight_frames").glob("*"))
    normal = list(Path(FRAMES_DIR+"/normal_frames").glob("*"))

    X = np.array([str(p) for p in violent + normal])
    y = np.array([1]*len(violent) + [0]*len(normal))

    print(f"Total violent videos: {len(violent)}")
    print(f"Total normal videos: {len(normal)}")
    print(f"Total videos: {len(X)}")

    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

    fold_accuracies = []
    fold_precisions = []
    fold_recalls = []
    fold_f1s = []

    for fold, (train_idx, test_idx) in enumerate(skf.split(X, y)):
        print(f"\n{'='*60}")
        print(f"FOLD {fold+1}/5")
        print(f"{'='*60}")

        X_train_fold = X[train_idx].tolist()
        X_test_fold = X[test_idx].tolist()
        y_train_fold = y[train_idx].tolist()
        y_test_fold = y[test_idx].tolist()

        # Split train into train/val (80-20)
        X_train, X_val, y_train, y_val = train_test_split(
            X_train_fold, y_train_fold, test_size=0.2, stratify=y_train_fold, random_state=42
        )

        print(f"Train: {len(X_train)} | Val: {len(X_val)} | Test: {len(X_test_fold)}")

        train_ds = ViolenceDataset(X_train, y_train)
        val_ds = ViolenceDataset(X_val, y_val)
        test_ds = ViolenceDataset(X_test_fold, y_test_fold)

        train_loader = DataLoader(train_ds, batch_size=4, shuffle=True)
        val_loader = DataLoader(val_ds, batch_size=4, shuffle=False)
        test_loader = DataLoader(test_ds, batch_size=4, shuffle=False)

        import timm
        model = MobileNetV3_ViT()

        best_state, best_val_acc = train_fold(model, train_loader, val_loader, device, epochs=20)

        model.load_state_dict(best_state)
        acc, prec, rec, f1, cm = evaluate_fold(model, test_loader, device)

        fold_accuracies.append(acc)
        fold_precisions.append(prec)
        fold_recalls.append(rec)
        fold_f1s.append(f1)

        print(f"\nFold {fold+1} Results:")
        print(f"  Accuracy: {acc:.4f} ({acc*100:.2f}%)")
        print(f"  Precision: {prec:.4f}")
        print(f"  Recall: {rec:.4f}")
        print(f"  F1-Score: {f1:.4f}")
        print(f"  Confusion Matrix: TN={cm[0,0]}, FP={cm[0,1]}, FN={cm[1,0]}, TP={cm[1,1]}")

    # ============================================
    # CROSS VALIDATION SUMMARY
    # ============================================

    print("\n" + "=" * 60)
    print("5-FOLD CROSS VALIDATION SUMMARY")
    print("=" * 60)

    mean_acc = np.mean(fold_accuracies)
    std_acc = np.std(fold_accuracies)
    mean_prec = np.mean(fold_precisions)
    mean_rec = np.mean(fold_recalls)
    mean_f1 = np.mean(fold_f1s)

    print(f"\nAccuracy: {mean_acc:.4f} ± {std_acc:.4f} ({mean_acc*100:.2f}% ± {std_acc*100:.2f}%)")
    print(f"Precision: {mean_prec:.4f} ± {np.std(fold_precisions):.4f}")
    print(f"Recall: {mean_rec:.4f} ± {np.std(fold_recalls):.4f}")
    print(f"F1-Score: {mean_f1:.4f} ± {np.std(fold_f1s):.4f}")

    print(f"\nPer-fold Accuracies: {[f'{acc:.4f}' for acc in fold_accuracies]}")

    print("\n" + "=" * 60)
    print("CROSS VALIDATION COMPLETE")
    print("=" * 60)

Using device: cuda
Total violent videos: 95
Total normal videos: 84
Total videos: 179

FOLD 1/5
Train: 114 | Val: 29 | Test: 36


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


model.safetensors:   0%|          | 0.00/22.1M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/346M [00:00<?, ?B/s]


Fold 1 Results:
  Accuracy: 0.9167 (91.67%)
  Precision: 0.9444
  Recall: 0.8947
  F1-Score: 0.9189
  Confusion Matrix: TN=16, FP=1, FN=2, TP=17

FOLD 2/5
Train: 114 | Val: 29 | Test: 36



Fold 2 Results:
  Accuracy: 0.8889 (88.89%)
  Precision: 0.8571
  Recall: 0.9474
  F1-Score: 0.9000
  Confusion Matrix: TN=14, FP=3, FN=1, TP=18

FOLD 3/5
Train: 114 | Val: 29 | Test: 36

Fold 3 Results:
  Accuracy: 0.9444 (94.44%)
  Precision: 0.9048
  Recall: 1.0000
  F1-Score: 0.9500
  Confusion Matrix: TN=15, FP=2, FN=0, TP=19

FOLD 4/5
Train: 114 | Val: 29 | Test: 36

Fold 4 Results:
  Accuracy: 0.9167 (91.67%)
  Precision: 0.9000
  Recall: 0.9474
  F1-Score: 0.9231
  Confusion Matrix: TN=15, FP=2, FN=1, TP=18

FOLD 5/5
Train: 115 | Val: 29 | Test: 35

Fold 5 Results:
  Accuracy: 0.9143 (91.43%)
  Precision: 0.9000
  Recall: 0.9474
  F1-Score: 0.9231
  Confusion Matrix: TN=14, FP=2, FN=1, TP=18

5-FOLD CROSS VALIDATION SUMMARY

Accuracy: 0.9162 ± 0.0176 (91.62% ± 1.76%)
Precision: 0.9013 ± 0.0277
Recall: 0.9474 ± 0.0333
F1-Score: 0.9230 ± 0.0160

Per-fold Accuracies: ['0.9167', '0.8889', '0.9444', '0.9167', '0.9143']

CROSS VALIDATION COMPLETE


Xception + ResNet101 + BiLSTM - 5-FOLD CROSS VALIDATION"

# MULTI-CNN FUSION - 5-FOLD CROSS VALIDATION

In [ ]:
import os
import numpy as np
import cv2
from pathlib import Path
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, roc_auc_score
from sklearn.naive_bayes import GaussianNB
import torch
import torch.nn as nn
import math
from torch.utils.data import Dataset, DataLoader
import torchvision.models as models
from torchvision import transforms

# ============================================
# METHOD 1: ResNet18 (as per paper Section 4.3.1, Fig 4)
# ============================================

class ResNet18(nn.Module):
    def __init__(self, num_classes=2):
        super(ResNet18, self).__init__()
        self.resnet18 = models.resnet18(pretrained=True)
        in_features = self.resnet18.fc.in_features
        self.resnet18.fc = nn.Linear(in_features, num_classes)

    def forward(self, x):
        return self.resnet18(x)


# ============================================
# METHOD 2: VGG16 (as per paper Section 4.3.2, Fig 5)
# ============================================

class VGG16(nn.Module):
    def __init__(self, num_classes=2):
        super(VGG16, self).__init__()
        self.vgg16 = models.vgg16(pretrained=True)
        in_features = self.vgg16.classifier[6].in_features
        self.vgg16.classifier[6] = nn.Linear(in_features, num_classes)

    def forward(self, x):
        return self.vgg16(x)


# ============================================
# WEIGHTED AVERAGING FUSION
# ============================================

class WeightedFusion:
    def __init__(self, resnet_acc, vgg16_acc):
        total = resnet_acc + vgg16_acc
        self.w1 = resnet_acc / total if total > 0 else 0.5
        self.w2 = vgg16_acc / total if total > 0 else 0.5

    def fuse(self, prob_resnet, prob_vgg16):
        return self.w1 * prob_resnet + self.w2 * prob_vgg16


# ============================================
# DATASET
# ============================================

class CartoonViolenceDataset(Dataset):
    def __init__(self, video_paths, labels, img_size=(100, 100)):
        self.video_paths = video_paths
        self.labels = labels
        self.img_size = img_size

        self.transform = transforms.Compose([
            transforms.ToPILImage(),
            transforms.Resize(img_size),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
        ])

    def __len__(self):
        return len(self.video_paths)

    def get_middle_frame(self, video_folder):
        video_path = Path(video_folder)
        frame_files = sorted(list(video_path.glob("*.jpg")) + list(video_path.glob("*.png")))
        if len(frame_files) == 0:
            return None
        middle_idx = len(frame_files) // 2
        return frame_files[middle_idx]

    def __getitem__(self, idx):
        video_path = self.video_paths[idx]
        frame_path = self.get_middle_frame(video_path)

        if frame_path is None:
            img = np.zeros((self.img_size[0], self.img_size[1], 3), dtype=np.uint8)
        else:
            img = cv2.imread(str(frame_path))
            if img is None:
                img = np.zeros((self.img_size[0], self.img_size[1], 3), dtype=np.uint8)
            else:
                img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

        img = self.transform(img)
        label = self.labels[idx]
        return img, torch.LongTensor([label])[0]


# ============================================
# TRAIN FUNCTIONS
# ============================================

def train_fold(model, train_loader, val_loader, epochs=30, lr=0.001, device='cuda'):
    model = model.to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)

    best_val_acc = 0
    best_state = None

    for epoch in range(epochs):
        model.train()
        train_loss = 0
        train_correct = 0
        train_total = 0

        for data, labels in train_loader:
            data, labels = data.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(data)
            loss = criterion(outputs, labels)

            if torch.isnan(loss):
                continue

            loss.backward()
            optimizer.step()

            train_loss += loss.item()
            _, predicted = torch.max(outputs.data, 1)
            train_total += labels.size(0)
            train_correct += (predicted == labels).sum().item()

        if train_total > 0:
            train_acc = 100 * train_correct / train_total
        else:
            train_acc = 0

        model.eval()
        val_loss = 0
        val_correct = 0
        val_total = 0

        with torch.no_grad():
            for data, labels in val_loader:
                data, labels = data.to(device), labels.to(device)
                outputs = model(data)
                loss = criterion(outputs, labels)

                if not torch.isnan(loss):
                    val_loss += loss.item()
                _, predicted = torch.max(outputs.data, 1)
                val_total += labels.size(0)
                val_correct += (predicted == labels).sum().item()

        if val_total > 0:
            val_acc = 100 * val_correct / val_total
        else:
            val_acc = 0

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}

    return best_state, best_val_acc


def evaluate_fold(model, test_loader, device='cuda'):
    model.eval()
    all_preds = []
    all_targets = []
    all_probs = []

    with torch.no_grad():
        for data, labels in test_loader:
            data, labels = data.to(device), labels.to(device)
            outputs = model(data)
            probs = torch.softmax(outputs, dim=1)

            _, predicted = torch.max(outputs.data, 1)

            all_preds.extend(predicted.cpu().numpy())
            all_targets.extend(labels.cpu().numpy())
            all_probs.extend(probs.cpu().numpy())

    if len(set(all_preds)) < 2:
        accuracy = accuracy_score(all_targets, all_preds)
        return accuracy, [0, 0], [0, 0], [0, 0], [[0,0],[0,0]], 0.5, all_probs

    accuracy = accuracy_score(all_targets, all_preds)
    precision = precision_score(all_targets, all_preds, average=None, zero_division=0)
    recall = recall_score(all_targets, all_preds, average=None, zero_division=0)
    f1 = f1_score(all_targets, all_preds, average=None, zero_division=0)
    cm = confusion_matrix(all_targets, all_preds)

    violent_probs = [p[1] for p in all_probs]
    try:
        auc = roc_auc_score(all_targets, violent_probs)
    except:
        auc = 0.5

    return accuracy, precision, recall, f1, cm, auc, all_probs


# ============================================
# VIOLENCE SCENE PATTERN
# ============================================

def discretize_violence_level(probabilities, num_levels=16):
    levels = []
    for prob in probabilities:
        level = int(prob * num_levels)
        level = min(level, num_levels)
        levels.append(level)
    return np.array(levels)


def create_temporal_sequences(violence_levels, window_size=5, prediction_horizon=1):
    X = []
    y = []
    for i in range(len(violence_levels) - window_size - prediction_horizon + 1):
        features = violence_levels[i:i+window_size]
        label = violence_levels[i+window_size+prediction_horizon-1]
        X.append(features)
        y.append(label)
    return np.array(X), np.array(y)


def train_naive_bayes(X_train, y_train, X_test, y_test, threshold=8):
    if len(X_train) == 0 or len(X_test) == 0:
        return None, 0, [0,0], [0,0], [0,0], [[0,0],[0,0]], 0.5

    nb_classifier = GaussianNB()
    nb_classifier.fit(X_train, y_train)
    y_pred = nb_classifier.predict(X_test)

    y_pred_binary = (y_pred > threshold).astype(int)
    y_test_binary = (y_test > threshold).astype(int)

    accuracy = accuracy_score(y_test_binary, y_pred_binary)
    precision = precision_score(y_test_binary, y_pred_binary, average=None, zero_division=0)
    recall = recall_score(y_test_binary, y_pred_binary, average=None, zero_division=0)
    f1 = f1_score(y_test_binary, y_pred_binary, average=None, zero_division=0)
    cm = confusion_matrix(y_test_binary, y_pred_binary)

    try:
        auc = roc_auc_score(y_test_binary, y_pred_binary)
    except:
        auc = 0.5

    return nb_classifier, accuracy, precision, recall, f1, cm, auc


# ============================================
# MAIN WITH 5-FOLD CV
# ============================================

if __name__ == "__main__":

    print("="*60)
    print("MULTI-CNN FUSION - 5-FOLD CROSS VALIDATION")
    print("="*60)

    IMG_SIZE = (224, 224)
    BATCH_SIZE = 8
    EPOCHS = 30
    LR = 0.001
    WINDOW_SIZE = 5
    PREDICTION_HORIZON = 1
    VIOLENCE_LEVELS = 16

    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"Using device: {device}")

    FRAMES_DIR = "/content/data/extracted_frames"

    pre_fight_dir = os.path.join(FRAMES_DIR, "pre_fight_frames")
    normal_dir = os.path.join(FRAMES_DIR, "normal_frames")

    if not os.path.exists(pre_fight_dir):
        print(f"Error: Directory not found - {pre_fight_dir}")
        exit()

    violent_videos = [str(p) for p in Path(pre_fight_dir).glob("*") if p.is_dir()]
    non_violent_videos = [str(p) for p in Path(normal_dir).glob("*") if p.is_dir()]

    print(f"\nFound {len(violent_videos)} violent videos")
    print(f"Found {len(non_violent_videos)} non-violent videos")

    if len(violent_videos) == 0 or len(non_violent_videos) == 0:
        print("Error: No videos found")
        exit()

    X = np.array(violent_videos + non_violent_videos)
    y = np.array([1] * len(violent_videos) + [0] * len(non_violent_videos))

    print(f"Total videos: {len(X)}")

    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

    fold_resnet_acc = []
    fold_vgg_acc = []
    fold_fusion_acc = []
    fold_fusion_auc = []

    all_fusion_probs = []
    all_fusion_labels = []

    for fold, (train_idx, test_idx) in enumerate(skf.split(X, y)):
        print(f"\n{'='*60}")
        print(f"FOLD {fold+1}/5")
        print(f"{'='*60}")

        X_train_fold = X[train_idx].tolist()
        X_test_fold = X[test_idx].tolist()
        y_train_fold = y[train_idx].tolist()
        y_test_fold = y[test_idx].tolist()

        X_train, X_val, y_train, y_val = train_test_split(
            X_train_fold, y_train_fold, test_size=0.2, stratify=y_train_fold, random_state=42
        )

        print(f"Train: {len(X_train)} | Val: {len(X_val)} | Test: {len(X_test_fold)}")

        train_dataset = CartoonViolenceDataset(X_train, y_train, IMG_SIZE)
        val_dataset = CartoonViolenceDataset(X_val, y_val, IMG_SIZE)
        test_dataset = CartoonViolenceDataset(X_test_fold, y_test_fold, IMG_SIZE)

        train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
        val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
        test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

        # ResNet18
        model_resnet = ResNet18(num_classes=2)
        best_state, best_val = train_fold(model_resnet, train_loader, val_loader, EPOCHS, LR, device)
        model_resnet.load_state_dict(best_state)
        acc_res, _, _, _, _, _, probs_res = evaluate_fold(model_resnet, test_loader, device)
        fold_resnet_acc.append(acc_res)

        # VGG16
        model_vgg = VGG16(num_classes=2)
        best_state, best_val = train_fold(model_vgg, train_loader, val_loader, EPOCHS, LR, device)
        model_vgg.load_state_dict(best_state)
        acc_vgg, _, _, _, _, _, probs_vgg = evaluate_fold(model_vgg, test_loader, device)
        fold_vgg_acc.append(acc_vgg)

        # Weighted Fusion
        fusion = WeightedFusion(acc_res, acc_vgg)
        fused_probs = []
        for prob_r, prob_v in zip(probs_res, probs_vgg):
            p_r = prob_r[1]
            p_v = prob_v[1]
            fused_prob = fusion.fuse(p_r, p_v)
            fused_probs.append(fused_prob)

        fused_preds = [1 if p > 0.5 else 0 for p in fused_probs]
        fused_acc = accuracy_score(y_test_fold, fused_preds)
        try:
            fused_auc = roc_auc_score(y_test_fold, fused_probs)
        except:
            fused_auc = 0.5

        fold_fusion_acc.append(fused_acc)
        fold_fusion_auc.append(fused_auc)
        all_fusion_probs.extend(fused_probs)
        all_fusion_labels.extend(y_test_fold)

        print(f"\nFold {fold+1} Results:")
        print(f"  ResNet18 Acc: {acc_res*100:.2f}%")
        print(f"  VGG16 Acc: {acc_vgg*100:.2f}%")
        print(f"  Fusion Acc: {fused_acc*100:.2f}%, AUC: {fused_auc:.4f}")

    # ============================================
    # CROSS VALIDATION SUMMARY
    # ============================================

    print("\n" + "=" * 60)
    print("5-FOLD CROSS VALIDATION SUMMARY")
    print("=" * 60)

    mean_res = np.mean(fold_resnet_acc)
    std_res = np.std(fold_resnet_acc)
    mean_vgg = np.mean(fold_vgg_acc)
    std_vgg = np.std(fold_vgg_acc)
    mean_fusion = np.mean(fold_fusion_acc)
    std_fusion = np.std(fold_fusion_acc)
    mean_fusion_auc = np.mean(fold_fusion_auc)
    std_fusion_auc = np.std(fold_fusion_auc)

    print(f"\nResNet18: {mean_res*100:.2f}% ± {std_res*100:.2f}%")
    print(f"VGG16: {mean_vgg*100:.2f}% ± {std_vgg*100:.2f}%")
    print(f"Weighted Fusion: {mean_fusion*100:.2f}% ± {std_fusion*100:.2f}%")
    print(f"Fusion AUC: {mean_fusion_auc:.4f} ± {std_fusion_auc:.4f}")

    print(f"\nPer-fold Fusion Accuracies: {[f'{acc*100:.2f}%' for acc in fold_fusion_acc]}")

    overall_auc = roc_auc_score(all_fusion_labels, all_fusion_probs) if len(set(all_fusion_labels)) > 1 else 0.5
    print(f"\nOverall Fusion AUC (all predictions): {overall_auc:.4f}")

    print("\n" + "=" * 60)
    print("CROSS VALIDATION COMPLETE")
    print("=" * 60)

MULTI-CNN FUSION - 5-FOLD CROSS VALIDATION
Using device: cuda

Found 95 violent videos
Found 84 non-violent videos
Total videos: 179

FOLD 1/5
Train: 114 | Val: 29 | Test: 36


/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet18_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet18_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 239MB/s]
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=VGG16_Weights.IMAGENET1K_V1`. You can also use `weights=VGG16_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Downloading: "https://download.pytorch.org/models/vgg16-397923af.pth" to /root/.cache/torch/hub/checkpoints/vgg16-397923af.pth


...


Fold 3 Results:
  ResNet18 Acc: 75.00%
  VGG16 Acc: 52.78%
  Fusion Acc: 72.22%, AUC: 0.7461

FOLD 4/5
Train: 114 | Val: 29 | Test: 36


/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet18_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet18_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are depreca


Fold 4 Results:
  ResNet18 Acc: 86.11%
  VGG16 Acc: 52.78%
  Fusion Acc: 86.11%, AUC: 0.8978

FOLD 5/5
Train: 115 | Val: 29 | Test: 35


/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet18_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet18_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are depreca


Fold 5 Results:
  ResNet18 Acc: 88.57%
  VGG16 Acc: 45.71%
  Fusion Acc: 80.00%, AUC: 0.8421

5-FOLD CROSS VALIDATION SUMMARY

ResNet18: 83.83% ± 4.71%
VGG16: 51.37% ± 2.83%
Weighted Fusion: 81.56% ± 5.18%
Fusion AUC: 0.8353 ± 0.0498

Per-fold Fusion Accuracies: ['86.11%', '83.33%', '72.22%', '86.11%', '80.00%']

Overall Fusion AUC (all predictions): 0.8570

CROSS VALIDATION COMPLETE


# MobBiLSTM (MobileNetV2 + BiLSTM)

In [ ]:
import os
import numpy as np
import cv2
from pathlib import Path
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, roc_auc_score
import torch
import torch.nn as nn
import math
from torch.utils.data import Dataset, DataLoader
import torchvision.models as models

# ============================================
# METHOD 1: MobBiLSTM
# ============================================

class MobBiLSTM(nn.Module):
    def __init__(self, num_classes=1):
        super(MobBiLSTM, self).__init__()
        self.mobilenetv2 = models.mobilenet_v2(pretrained=True)
        self.mobilenetv2.classifier = nn.Identity()
        self.bilstm = nn.LSTM(1280, 256, num_layers=2, batch_first=True, bidirectional=True)
        self.classifier = nn.Linear(512, num_classes)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        b, t, c, h, w = x.shape
        x = x.view(b * t, c, h, w)
        feat = self.mobilenetv2(x)
        feat = feat.view(b, t, -1)
        lstm_out, _ = self.bilstm(feat)
        out = self.classifier(lstm_out[:, -1, :])
        out = self.sigmoid(out)
        return out   # FIXED


# ============================================
# METHOD 2: MobileTransformerSeq
# ============================================

class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=500):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        pos = torch.arange(0, max_len).unsqueeze(1).float()
        div = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(pos * div)
        pe[:, 1::2] = torch.cos(pos * div)
        self.register_buffer('pe', pe.unsqueeze(0))

    def forward(self, x):
        return x + self.pe[:, :x.size(1)]


class MobileTransformerSeq(nn.Module):
    def __init__(self, num_classes=1, d_model=256, nhead=8, num_layers=4):
        super().__init__()
        self.mobilenetv2 = models.mobilenet_v2(pretrained=True)
        self.mobilenetv2.classifier = nn.Identity()
        self.proj = nn.Linear(1280, d_model)
        self.pos = PositionalEncoding(d_model)

        enc_layer = nn.TransformerEncoderLayer(d_model, nhead, 512, batch_first=True)
        self.transformer = nn.TransformerEncoder(enc_layer, num_layers)

        self.classifier = nn.Linear(d_model, num_classes)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        b, t, c, h, w = x.shape
        x = x.view(b * t, c, h, w)
        feat = self.mobilenetv2(x)
        feat = feat.view(b, t, -1)
        feat = self.proj(feat)
        feat = self.pos(feat)

        temp = self.transformer(feat)
        out = self.classifier(temp[:, -1, :])
        out = self.sigmoid(out)
        return out   # FIXED


# ============================================
# DATASET
# ============================================

class ViolenceDataset(Dataset):
    def __init__(self, videos, labels, seq_len=16, img_size=(224,224)):
        self.videos = videos
        self.labels = labels
        self.seq_len = seq_len
        self.img_size = img_size

    def load_video(self, folder):
        frames = []
        files = sorted(list(Path(folder).glob("*.jpg")))
        total = len(files)

        if total >= self.seq_len:
            idx = np.linspace(0, total-1, self.seq_len, dtype=int)
            files = [files[i] for i in idx]

        for f in files:
            img = cv2.imread(str(f))
            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            img = cv2.resize(img, self.img_size)
            img = img / 255.0
            frames.append(img)

        while len(frames) < self.seq_len:
            frames.append(np.zeros((224,224,3)))

        return np.array(frames, dtype=np.float32)

    def __len__(self):
        return len(self.videos)

    def __getitem__(self, idx):
        x = self.load_video(self.videos[idx])
        y = self.labels[idx]

        x = torch.tensor(x).permute(0,3,1,2)

        mean = torch.tensor([0.485,0.456,0.406]).view(1,3,1,1)
        std = torch.tensor([0.229,0.224,0.225]).view(1,3,1,1)
        x = (x - mean) / std

        y = torch.tensor(y, dtype=torch.float32).view(1)  # FIXED

        return x, y


# ============================================
# TRAIN
# ============================================

def train_fold(model, train_loader, val_loader, epochs=30, lr=1e-3, device='cuda'):
    model.to(device)
    crit = nn.BCELoss()
    opt = torch.optim.Adam(model.parameters(), lr=lr)

    best_acc = 0
    best_state = None

    for _ in range(epochs):
        model.train()
        for x,y in train_loader:
            x,y = x.to(device), y.to(device)
            opt.zero_grad()
            out = model(x)
            loss = crit(out, y)
            loss.backward()
            opt.step()

        model.eval()
        correct,total = 0,0
        with torch.no_grad():
            for x,y in val_loader:
                x,y = x.to(device), y.to(device)
                out = model(x)
                pred = (out>0.5).float()
                total += y.size(0)
                correct += (pred==y).sum().item()

        acc = correct/total
        if acc > best_acc:
            best_acc = acc
            best_state = {k:v.cpu() for k,v in model.state_dict().items()}

    return best_state, best_acc


# ============================================
# EVALUATE
# ============================================

def evaluate_fold(model, loader, device='cuda'):
    model.eval()
    preds, probs, targets = [], [], []

    with torch.no_grad():
        for x,y in loader:
            x,y = x.to(device), y.to(device)
            out = model(x)

            probs.extend(out.cpu().numpy().flatten())
            preds.extend((out>0.5).cpu().numpy().flatten())
            targets.extend(y.cpu().numpy().flatten())

    return accuracy_score(targets, preds), roc_auc_score(targets, probs)


# ============================================
# MAIN
# ============================================

if __name__ == "__main__":
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    FRAMES_DIR = "/content/data/extracted_frames"
    v = [str(p) for p in Path(FRAMES_DIR+"/pre_fight_frames").glob("*")]
    n = [str(p) for p in Path(FRAMES_DIR+"/normal_frames").glob("*")]

    X = np.array(v+n)
    y = np.array([1]*len(v) + [0]*len(n))

    skf = StratifiedKFold(5, shuffle=True, random_state=42)

    for fold,(tr,te) in enumerate(skf.split(X,y)):
        X_tr, X_te = X[tr], X[te]
        y_tr, y_te = y[tr], y[te]

        X_tr, X_val, y_tr, y_val = train_test_split(X_tr, y_tr, test_size=0.2, stratify=y_tr)

        train_loader = DataLoader(ViolenceDataset(X_tr,y_tr), batch_size=4, shuffle=True)
        val_loader   = DataLoader(ViolenceDataset(X_val,y_val), batch_size=4)
        test_loader  = DataLoader(ViolenceDataset(X_te,y_te), batch_size=4)

        m1 = MobBiLSTM()
        s1,_ = train_fold(m1, train_loader, val_loader, device=device)
        m1.load_state_dict(s1)
        acc1,auc1 = evaluate_fold(m1, test_loader, device)

        m2 = MobileTransformerSeq()
        s2,_ = train_fold(m2, train_loader, val_loader, device=device)
        m2.load_state_dict(s2)
        acc2,auc2 = evaluate_fold(m2, test_loader, device)

        print(f"Fold {fold+1}")
        print(f"MobBiLSTM: Acc={acc1:.4f}, AUC={auc1:.4f}")
        print(f"Transformer: Acc={acc2:.4f}, AUC={auc2:.4f}")

/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=MobileNet_V2_Weights.IMAGENET1K_V1`. You can also use `weights=MobileNet_V2_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are

Fold 1
MobBiLSTM: Acc=0.8611, AUC=0.9659
Transformer: Acc=0.5278, AUC=0.7523


/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=MobileNet_V2_Weights.IMAGENET1K_V1`. You can also use `weights=MobileNet_V2_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are

Fold 2
MobBiLSTM: Acc=0.8889, AUC=0.8978
Transformer: Acc=0.6389, AUC=0.8050


/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=MobileNet_V2_Weights.IMAGENET1K_V1`. You can also use `weights=MobileNet_V2_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are

Fold 3
MobBiLSTM: Acc=0.8889, AUC=0.9505
Transformer: Acc=0.5278, AUC=0.7368


/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=MobileNet_V2_Weights.IMAGENET1K_V1`. You can also use `weights=MobileNet_V2_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are

Fold 4
MobBiLSTM: Acc=0.9167, AUC=0.9814
Transformer: Acc=0.5278, AUC=0.7399


/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=MobileNet_V2_Weights.IMAGENET1K_V1`. You can also use `weights=MobileNet_V2_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are

Fold 5
MobBiLSTM: Acc=1.0000, AUC=1.0000
Transformer: Acc=0.5429, AUC=0.7566
